# Introduction: Hospital Surge Planning with BI and Q-Learning

The goal of this recitation is to show how sequential decision problems can be solved with different tools. We focus on a hospital surge staffing example: the hospital must decide how many additional doctors to call in each hour, balancing staffing cost against patient waiting cost. The system evolves over time, so the decision this hour affects next hour's beginning state. This setting naturally fits into the framework of sequential decision-making under uncertainty.

Two methods can be applied to this problem: **backward induction (BI)** from dynamic programming and **Q-learning (QL)** from reinforcement learning. Both aim to find a policy that minimizes total cost, but they differ in what knowledge is assumed and how the policy is derived.

**Backward Induction (BI).**  
BI assumes the hospital manager (the agent, the decision maker) knows the full transition and cost functions in advance. For example, the patient queue evolves as  
$$
x_{t+1} = \max\{0, x_t + d_t - 2(10+u_t)\},
$$  
and the per-period cost is  
$$
g_t(x_t,u_t) = w \cdot x_t + p \cdot u_t.
$$  
With this information, the Bellman recursion  
$$
J_t(x_t) = \min_{u_t \ge 0} \{ g_t(x_t,u_t) + J_{t+1}(x_{t+1}) \}
$$  
yields the exact optimal policy. BI provides the benchmark solution when the model is known.

**Q-Learning (QL).**  
QL does not assume such knowledge (e.g. transition function, demands information). Instead, the manager only observes outcomes by trying actions in the environment. The Q-value table is updated through experience using  
$$
Q_t(s,a) \leftarrow (1-\alpha)Q_t(s,a) + \alpha \big(r + \gamma \max_{a'} Q_t(s',a')\big).
$$  
Over repeated episodes, the Q-values converge toward the optimal values, and the derived policy approaches the BI benchmark. QL reflects a “learning by doing” process and is especially useful when the transition or cost functions are unknown.

**Comparison.**  
- **Usage:** BI requires system knowledge and computes the exact optimal policy; QL learns from feedback without needing the model.  
- **Results:** BI produces the true optimal cost and policy if the model is accurate; QL may be less efficient early on but can converge to near-optimal performance through exploration and learning.

This recitation highlights how the same hospital surge problem can be addressed through both a model-based approach (BI) and a model-free approach (QL), illustrating the trade-offs between knowledge, computation, and learning.




## Part I - Hospital Surge Scheduling with Backward Induction

This part implements **backward induction** for a stylized hospital/clinic surge-staffing problem.

**You will:**
1. Build the dynamic program for a 12‑hour operating day (8am–8pm).
2. Compute the **optimal surge policy** $u_t^*$ for each hour $t=1,\dots,12$ and each queue length $x \in \{0,\dots,70\}$.
3. Output the policy (table + heatmap) and the **cost‑to‑go** $J_t(x)$ (table + heatmap).
4. Run a **sensitivity analysis** over waiting cost $w$ and abandonment/terminal cost $W$.

**Parameters:**
- Horizon: $T=12$ hours (8am–8pm).
- State $x_t$: patients waiting at start of hour $t$.
- Action $u_t \in \mathbb{Z}_{\ge 0}$: number of **on‑demand** doctors to add in hour $t$.
- Base doctors: $10$ permanently scheduled (not part of control; cost sunk).
- Service: each doctor (base or on‑demand) treats **2 patients/hour**.
- Arrivals: at the beginning of each hour, a random $D_t \sim \text{Poisson}(\lambda)$ with $\lambda=17$.
- Per‑hour **waiting cost** $w=30$ per waiting patient.
- **On‑demand wage** $p=500$ per extra doctor per hour.
- **Terminal cost** $W=400$ per patient left at end of day.

**Transition Function**
$$x_{t+1}(x_t, D_t, u_t) = \max\{ 0,\; x_t + D_t - 2(10+u_t) \}$$

**Cost Function**
$$\begin{aligned}
g_t(x_t,u_t) = w\,x_t + p\,u_t, \quad
J_{T+1}(x_{T+1}) = W\,x_{T+1}.
\end{aligned}$$

**Bellman recursion (expectation over arrivals)**
$$\begin{aligned}
J_t(x_t) = \min_{u_t \in \mathbb{Z}_{\ge 0}} \Big[ w\,x_t + p\,u_t + \mathbb{E}_{D_t \sim \text{Poi}(\lambda)} \big(J_{t+1}(x_{t+1}(x_t, D_t, u_t))\big) \Big].
\end{aligned}$$

We will solve this by discretizing $x_t$ to $\{0,\dots, X_{\max}\}$ with $X_{\max}=70$ (given) and truncating the Poisson support 
to a practical set $\{0,\dots,D_{\max}\}$, where $D_{\max}=20$.


---
**How the Cost-to-Go Table is Updated**

To better understand backward induction, let us look at how the **cost-to-go table** $J_t(x)$ is constructed step by step.  

At the **terminal stage** ($t = T+1$), the cost-to-go is simply the terminal penalty:
$$
J_{T+1}(x) = W \cdot x.
$$

For example, if $W=400$, then
| $x_{13}$ (patients left) | $J_{13}(x)$ |
|------------------------|-------------|
| 0                      | 0           |
| 1                      | 400         |
| 2                      | 800         |
| 3                      | 1200        |
|$\vdots$                | $\vdots$    |

Now, at **hour $t=T=12$**, suppose $w=30$, $p=500$, and we approximate arrivals by $D_{12}\in\{0,1,2\}$ with equal probabilities for illustration. The Bellman recursion is
$$
J_{12}(x_{12}) = \min_{u_{12}\ge 0} \Big[ w x_{12} + p u_{12} + \mathbb{E}\big(J_{13}(x_{13}(x_{12}, D_{12}, u_{12}))\big) \Big].
$$

For example, take state $x_{12}=2$ (2 patients waiting at start of hour 12):  
- If $u_{12}=0$ (no surge doctor), capacity = 20 patients, then $x_{13}=\max\{0,2+D_{12}-20\}=0$ for all $D_{12}$.  
  Total cost = $30\cdot 2 + 500\cdot 0 + \mathbb{E}[J_{13}(0)] = 60 + 0 + 0 = 60$.  
- If $u_{12}=1$, capacity = 22 patients, then still $x_{13}=0$.  
  Total cost = $30\cdot 2 + 500\cdot 1 + 0 = 560$. 

So the minimum is at $u_{12}=0$ and $J_{12}(2)=60$.  

Similarly, at $x_{12}=19$:  
- With $u_{12}=0$, leftover $x_{13}$ can be positive if $D_{12}=2$. Then the expected next stage cost $\mathbb{E}(J_{13}(x_{13}))$ is $400/3\approx 133.3$, then $J_{12}(x_{12}=19)=30\cdot 19 + 0 + 133.3 = 703.3$.  
- With $u_{12}=1$, leftover $x_{13}$ will always be zero because $D_{12}\leq 2$, but the service capacity is $22$. Then the expected next stage cost $\mathbb{E}(J_{13}(x_{13}))=0$, and $J_{12}(x_{12}=19)=30\cdot 19 + 500 + 0 = 1070$.  
Here the policy may prefer $u_{12}=0$, but for larger $x_{12}$, larger $u_{12}=1$ may become optimal.

So the table for $J_{12}(x)$ begins to fill in like:

| $x_t$    | $J_{12}(x)$ after update| $J_{13}(x)$ (terminal) |
|--------|-------------------------|------------------------|
| 0      | 0                       | 0                      |
|$\vdots$| $\vdots$                | $\vdots$               |
| 2      | **60**                  | 800                    |
|$\vdots$| $\vdots$                | $\vdots$               |
| 19     | **703.3**               | 7600                   |
|$\vdots$| $\vdots$                | $\vdots$               |

The recursion continues backward to $t=11,10,\dots,1$, each time using $J_{t+1}(\cdot)$ to update $J_t(\cdot)$. This step-by-step update explains how the dynamic program builds the full cost-to-go table and the optimal policy.

---

**Now we implement the backward induction**

In [1]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt

# Use matplotlib only; do not set custom colors/styles per instructions.
plt.rcParams["figure.dpi"] = 120
rng = np.random.default_rng(42)

def poisson_pmf_trunc(lam=5.0, d_max=20):
    """Return normalized probabilities for D=0..d_max and the leftover (truncation) mass."""
    probs = np.array([math.exp(-lam) * lam**d / math.factorial(d) for d in range(d_max+1)], dtype=float)
    mass = probs.sum()
    if mass <= 0:
        probs = np.zeros(d_max+1, dtype=float)
        probs[0] = 1.0
        mass = 1.0
    return probs / mass, 1.0 - mass


def build_dp(
    T=12,      # number of hours (stages)
    x_max=70,  # maximum queue length (state)
    lam=5.0,   # Poisson arrival rate per hour
    d_max=20,  # truncate Poisson support to {0..d_max}
    w=200.0,   # waiting cost per patient per hour
    W=400.0,   # terminal cost per patient
    p=500.0,   # on-demand doctor wage per hour
    u_max=None # optional cap on action (if None, compute a safe cap per state)
):
    """
    Solve the DP by backward induction with expectation over Poisson arrivals.
    Returns:
        J (np.ndarray): shape (T+2, x_max+1) cost-to-go; index t in [1..T+1], x in [0..x_max].
        U (np.ndarray): shape (T+1, x_max+1) optimal action table; U[t,x] is u*_t(x).
    """
    pdist, _ = poisson_pmf_trunc(lam=lam, d_max=d_max)
    D_support = np.arange(d_max+1, dtype=int)

    J = np.full((T+2, x_max+1), np.inf, dtype=float)  # cost-to-go
    U = np.zeros((T+1, x_max+1), dtype=int)          # optimal action

    # Terminal cost
    J[T+1, :] = W * np.arange(x_max+1, dtype=float)

    # Backward induction
    for t in range(T, 0, -1):
        for x in range(x_max+1):
            # Dynamic action cap: ensure we don't search an excessive range
            if u_max is None:
                # enough to clear worst-case (x + d_max) with 20 + 2u capacity
                u_cap = max(0, math.ceil((x + d_max - 20) / 2))
                u_cap = min(u_cap, 60)  # hard cap
            else:
                u_cap = u_max

            best_val = np.inf
            best_u = 0
            for u in range(u_cap + 1):
                # YOUR CODE HERE
                # compute total cost = immediate cost + expected future cost
                imm = w * x + p * u
                cap = 2 * (10 + u)

                x_next = x + D_support - cap
                x_next = np.clip(x_next, 0, x_max)

                exp_future = float(np.dot(pdist, J[t+1, x_next]))
                total = imm + exp_future
                # End YOUR CODE HERE

                if total < best_val:
                    best_val = total
                    best_u = u

            J[t, x] = best_val
            U[t, x] = best_u

    return J, U


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    a


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    a


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    a


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/pyzhang/opt/anaconda3/lib/python3.9/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    a

In [2]:
# Default parameters from the assignment
T = 12
X_MAX = 70
LAM = 17.0
D_MAX = 20
W = 400.0
w = 30.0
WAGE = 500.0
J, U = build_dp(T=T, x_max=X_MAX, lam=LAM, d_max=D_MAX, w=w, W=W, p=WAGE)
print("Solved DP. Shapes:", J.shape, U.shape)

Solved DP. Shapes: (14, 71) (13, 71)



### Optimal Policy Tables and Heatmaps of BI

- **Policy table**: rows = state $x_t \in \{0,\dots,70\}$, columns = hour $t=1,\dots,12$, entries = optimal on‑demand doctors $u_t^*(x_t)$.
- **Cost‑to‑go table**: rows = $x_t \in \{0,\dots,70\}$, columns = $t=1,\dots,12$, entries = $J_t(x_t)$.
- **Heatmaps**: visual versions of both tables.


In [3]:
states = np.arange(X_MAX+1)
hours  = np.arange(1, T+1)

BI_policy_df = pd.DataFrame(U[1:T+1, :].T, index=states, columns=hours)  # shape (x, t)
BI_cost_df   = pd.DataFrame(J[1:T+1, :].T, index=states, columns=hours)  # shape (x, t)

# Show heads to confirm structure
display(BI_policy_df.head(3))
display(BI_cost_df.head(3))


,1,2,3,4,5,6,7,8,9,10,11,12
0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0


,1,2,3,4,5,6,7,8,9,10,11,12
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,32.817742,32.817742,32.817742,32.817742,32.817743,32.817755,32.817889,32.819456,32.837697,33.050155,35.524609,64.344133
2,69.261835,69.261835,69.261835,69.261836,69.261847,69.261964,69.263166,69.275323,69.395454,70.544644,81.017932,169.093127


In [4]:
plt.figure(figsize=(8, 5))
plt.imshow(BI_policy_df.values, aspect='auto', origin='lower')
plt.colorbar()
plt.yticks(ticks=np.linspace(0, X_MAX, 8), labels=[str(int(v)) for v in np.linspace(0, X_MAX, 8)])
plt.xticks(ticks=np.arange(T), labels=[str(t) for t in range(1, T+1)])
plt.xlabel("Hour t (1 = 8–9am, …, 12 = 7–8pm)")
plt.ylabel("Queue length x at start of hour")
plt.title("Optimal surge policy u*_t(x)")
plt.show()


/var/folders/70/8g8pbwhx56z6cbq_qh7gxzbm0000gn/T/ipykernel_90283/3987032760.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
plt.figure(figsize=(8, 5))
plt.imshow(BI_cost_df.values, aspect='auto', origin='lower')
plt.colorbar()
plt.yticks(ticks=np.linspace(0, X_MAX, 8), labels=[str(int(v)) for v in np.linspace(0, X_MAX, 8)])
plt.xticks(ticks=np.arange(T), labels=[str(t) for t in range(1, T+1)])
plt.xlabel("Hour t (1 = 8–9am, …, 12 = 7–8pm)")
plt.ylabel("Queue length x at start of hour")
plt.title("Cost-to-go J_t(x)")
plt.show()

/var/folders/70/8g8pbwhx56z6cbq_qh7gxzbm0000gn/T/ipykernel_90283/2610654412.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



## Sensitivity Analysis: Varying $w$ and $W$

**Intuition**
- Larger **waiting cost** $w$ makes the system more averse to carrying a queue within the day, so it tends to **hire more surge earlier** to drain the queue.
- Larger **terminal cost** $W$ makes end‑of‑day backlog very expensive, encouraging **aggressive staffing near the end** to avoid leftover patients.

Below we sweep a few $(w, W)$ pairs and compare the **policy in the first and last hours**, which usually shows the clearest shift.


In [6]:

def summarize_policy_for(w, W):
    J_tmp, U_tmp = build_dp(T=T, x_max=X_MAX, lam=LAM, d_max=D_MAX, w=w, W=W, p=WAGE)
    return U_tmp

pairs = [
    (100.0, 200.0),
    (200.0, 400.0),  # baseline
    (300.0, 400.0),
    (200.0, 800.0),
]

summaries = []
for w_val, W_val in pairs:
    U_tmp = summarize_policy_for(w_val, W_val)
    summaries.append({
        "w": w_val, "W": W_val,
        "u*_t=1(x=0)": int(U_tmp[1,0]),
        "u*_t=1(x=30)": int(U_tmp[1,30]),
        "u*_t=1(x=60)": int(U_tmp[1,60]),
        "u*_t=12(x=0)": int(U_tmp[12,0]),
        "u*_t=12(x=30)": int(U_tmp[12,30]),
        "u*_t=12(x=60)": int(U_tmp[12,60]),
    })

pd.DataFrame(summaries)


,w,W,u*_t=1(x=0),u*_t=1(x=30),u*_t=1(x=60),u*_t=12(x=0),u*_t=12(x=30),u*_t=12(x=60)
0,100.0,200.0,0,9,24,0,0,0
1,200.0,400.0,0,12,27,0,12,27
2,300.0,400.0,0,13,28,0,12,27
3,200.0,800.0,0,12,27,0,14,29



**Qualitative takeaways**
- Increasing $w$ (holding $W$ fixed) raises the **penalty of within‑day queueing**, shifting the policy upward across many hours—more surge for the same $x$.
- Increasing $W$ (holding $w$ fixed) has the strongest effect **toward the end of the day**—surge is added to avoid expensive leftovers.
- When both $w$ and $W$ are low, the policy is conservative (hire later/less). When both are high, the policy is aggressive (hire earlier/more).

*Tip:* Adjust the `pairs` list or visualize full heatmaps for each pair to explore further.'

**How to Use**
- Change any of `w`, `W`, `WAGE`, `LAM`, `D_MAX`, or `X_MAX` above and re‑run.
- The Poisson support is **truncated** at `D_MAX` with re‑normalization.
- All figures use Matplotlib only (no seaborn; no explicit color settings).


## Part II: Q-Learning in the Hospital Environment

In the last part, we used **Backward Induction (BI)** to solve the hospital scheduling problem.  
While BI gives the exact optimal policy, it has two major limitations:

* **Requires a known transition model.**  
  BI relies on the agent (decision maker) knowing the patient arrival distribution and the exact transition function. In reality, the model could be hard to obtain or entirely unknown, and the agent can only observe the feedback from the environment.

* **Computationally expensive.**  
  BI computes the optimal action for *all* states at *all* stages, even if many states are unlikely to be visited. Practically, we would prefer to spend computation on states that actually occur.

**Q-Learning (QL)** is a reinforcement learning algorithm that addresses these issues:

* It is **model-free** — the agent learns from interactions without needing the transition law or arrival distribution.  
* It focuses on the state-action pairs actually visited, making it more efficient in large or complex environments.

In this part, we will implement **Q-Learning** to solve the same hospital surge staffing problem and obtain a near-optimal policy.


#### The Q-Learning Environment

The hospital environment is already finished. Note the agent does not know how this environment works, but can only observes its feedback on actions. A few necessary functions are:

* A **reset()** function to start each episode from an initial state $(t=1, x_1)$.  
* A **step()** function to take an action $u_t$, advance to $(t+1, x_{t+1})$, and return the cost $c_t$ and whether the episode has finished.  
* At the terminal stage $(t=T+1)$, the environment returns the terminal cost $r_{T+1} = W x_{T+1}$.

In [7]:
class HospitalEnvQL:
    def __init__(self, T=12, X_MAX=80, U_MAX=10, lam=17.0, w=30.0, W=300.0, wage=500.0, base_docs=10, service_per_doc=2):
        self.T = T
        self.X_MAX = X_MAX
        self.U_MAX = U_MAX
        self.lam = lam
        self.w = w
        self.W = W
        self.wage = wage
        self.base_docs = base_docs
        self.service_per_doc = service_per_doc
        self._t = None
        self._x = None

    def reset(self, x0=None):
        self._t = 1
        self._x = int(rng.integers(0, self.X_MAX+1)) if x0 is None else int(np.clip(x0, 0, self.X_MAX))
        return (self._t, self._x)

    def step(self, u):
        t, x = self._t, self._x
        u = int(np.clip(u, 0, self.U_MAX))
        cost = self.w * x + self.wage * u
        cap = self.service_per_doc * (self.base_docs + u)
        d = rng.poisson(self.lam)
        x_next = int(np.clip(x + d - cap, 0, self.X_MAX))
        t_next = t + 1
        done = False
        if t_next > self.T:
            term_cost = self.W * x_next
            cost += term_cost
            done = True
            next_state = (self.T + 1, x_next)
        else:
            next_state = (t_next, x_next)
        self._t, self._x = next_state
        return next_state, cost, done


#### The Q-Learning Agent

We then build the agent to make decisions based on Q-learning and $\epsilon$-greedy rule.

* **Q-Function:**  
  Recall the cost-to-go function from BI:
  $$
  J_t(x_t) = \min_{u_t} \bigl\{ g_t(x_t,u_t) + \mathbb{E}[ J_{t+1}(f_t(x_t,u_t)) ] \bigr\}
  $$

  Q-Learning expands this by considering the action $u_t$:  
  $$
  Q_t(x_t, u_t) = g_t(x_t,u_t) + \mathbb{E}[ J_{t+1}(f_t(x_t,u_t)) ]
  $$

  Here, `Q[t][x_t][u_t]` stores the Q-value of taking action $u_t$ at queue length $x_t$ at stage $t$.

* **Policy (ε-greedy):**  
  $$
  \pi(x_t) = 
  \begin{cases}
  \text{random action} & \text{with prob. } \epsilon, \\[6pt]
  \arg\min_{u_t} Q_t(x_t, u_t) & \text{with prob. } 1-\epsilon.
  \end{cases}
  $$
  Exploration is controlled by $\epsilon$, which decays over time.

* **Q-update rule:**  
  $$
  Q_t(x_t, u_t) \leftarrow (1-\alpha) Q(x_t,u_t) + \alpha \Bigl( r_t + \gamma \min_{u_{t+1}} Q_{t+1}(x_{t+1}, u_{t+1}) \Bigr).
  $$  
  - Learning rate $\alpha$ controls how much new info is trusted.  
  - Discount factor $\gamma$ weights future costs. For this finite-horizon hospital case, $\gamma = 1$.

#### Finite-Horizon Q-Learning Algorithm (Minimization with Costs)

1. Initialize $Q_t(x_t,u_t)$ for all stages $t=1,\dots,T$, states $x_t$, and actions $u_t$  
2. **for** each episode **do**  
3. &nbsp;&nbsp;&nbsp;&nbsp;Initialize state $x_1$ at $t=1$  
4. &nbsp;&nbsp;&nbsp;&nbsp;**while** $t \le T$ **do**  
5. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Choose action $u_t$ (ε-greedy, UCB, softmax, …)  
6. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Take action $u_t$, observe cost $c_t$ and next state $x_{t+1}$  
7. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**Update Q-table:**  

   New value ← $(1-\alpha)\cdot \text{Old Value} + \alpha \cdot \text{New Observation}$  

   $$
   Q_t(x_t,u_t) \;\leftarrow\; (1-\alpha)\,Q_t(x_t,u_t) \;+\; 
   \alpha\Big(c_t + \gamma \min_{u_{t+1}} Q_{t+1}(x_{t+1},u_{t+1})\Big)
   $$  

   with terminal condition $Q_{T+1}(x_{T+1},u_{T+1}) \equiv 0$.  

8. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Set $x_t \leftarrow x_{t+1}$ and $t \leftarrow t+1$  
9. &nbsp;&nbsp;&nbsp;&nbsp;**end while**  
10. **end for**

---

**Note.**  
This explicit time-indexed formulation is common for finite-horizon problems like hospital surge staffing. An equivalent way to write it is to treat the pair $(t,x_t)$ as the *augmented state*. Then the update rule takes the familiar stationary Q-learning form
$$
Q((t,x_t),u_t) \;\leftarrow\; (1-\alpha)Q((t,x_t),u_t) + \alpha\Big(c_t + \gamma \min_{u_{t+1}} Q_{t+1}((t+1,x_{t+1}),u_{t+1})\Big).
$$
This shows the connection between finite-horizon Q-learning and the standard infinite-horizon formulation students may have seen before.

---

**How the Q-Table is Updated**

Different from backward induction updated the cost-to-go table stage by stage, Q-learning updates the Q-table **incrementally** from experience. Each running the 12-hour operations with one set of $u_{t}, t=1,\cdots, 12$ can be regarded as one experience (or episode).

Suppose we are at stage $t=5$ with queue length $x_5=3$ patients, and we take action $u_5=1$ (hire one surge doctor). The environment responds with cost $c_5=600$ and next state $x_6=4$. With learning rate $\alpha=0.3$ and $\gamma=1$, the update is

$$
Q_5(3,1) \;\leftarrow\; (1-\alpha)Q_5(3,1) + \alpha \Big( c_5 + \min_{u_6} Q_6(4,u_6) \Big).
$$

**Example Q-Table Before Update**
| Old $Q_t(x_t,u)$ | $u=0$  |  $u=1$ |  $u=2$ |$\cdots$|
|------------------|--------|--------|--------|--------|
|$\vdots$          |$\vdots$|$\vdots$|$\vdots$|$\vdots$|
| $(t,x_t) =(5,3)$ |1800    |**2000**| 2200   |$\cdots$|
|$\vdots$          |$\vdots$|$\vdots$|$\vdots$|$\vdots$|
| $(t,x_t) =(6,4)$ |2100    | 1500   |3000    |$\cdots$|
|$\vdots$          |$\vdots$|$\vdots$|$\vdots$|$\vdots$|

Suppose the best continuation cost at $(t=6,x_6=4)$ is $\min_{u_6} Q_6(4,u_6) = 1500$. Then

$$
\text{Target} = c_5 + 1500 = 600 + 1500 = 2100.
$$

The updated Q-value is

$$
Q_5(3,1) \;\leftarrow\; 0.7\cdot 2000 + 0.3\cdot 2100 = 2030.
$$

**Example Table After Update**
| New $Q_t(x_t,u)$ | $u=0$  |  $u=1$ |  $u=2$ |$\cdots$|
|------------------|--------|--------|--------|--------|
|$\vdots$          |$\vdots$|$\vdots$|$\vdots$|$\vdots$|
| $(t,x_t) =(5,3)$ |1800    |**2030**| 2200   |$\cdots$|
|$\vdots$          |$\vdots$|$\vdots$|$\vdots$|$\vdots$|
| $(t,x_t) =(6,4)$ |2100    | 1500   |3000    |$\cdots$|
|$\vdots$          |$\vdots$|$\vdots$|$\vdots$|$\vdots$|

This shows how Q-learning uses **sample feedback** to gradually refine the Q-table. Over many episodes, these updates drive $Q_t(x_t,u_t)$ toward the optimal values.

---

**Clarification:**  
- **Q-table** $Q_t(x_t,u_t)$: expected cumulative cost of taking action $u_t$ in state $x_t$ at stage $t$, then following the current best policy.  
- **Policy table** $\pi_t(x_t)$: the best action at each state and time, derived from the Q-table (e.g. $\pi_t(x_t)=\arg\min_{u_t}Q_t(x_t,u_t)$).  
- **Cost-to-go table** $J_t(x_t)$: the expected optimal cumulative cost from state $x_t$ at stage $t$, equal to $\min_{u_t} Q_t(x_t,u_t)$.

---



**Now we build the Q-learning agent, who does not how the costs occur and the state transits**

In [8]:
class QLearningAgent:
    def __init__(self, T, X_MAX, U_MAX, gamma=1.0, epsilon=1.0, eps_min=0.02, eps_dec=2e-4,
                 lr=0.3, lr_min=0.05, lr_dec=5e-7):
        self.T = T
        self.X_MAX = X_MAX
        self.U_MAX = U_MAX
        self.gamma = gamma
        self.epsilon = float(epsilon)
        self.eps_min = float(eps_min)
        self.eps_dec = float(eps_dec)
        self.lr = float(lr)
        self.lr_min = float(lr_min)
        self.lr_dec = float(lr_dec)
        self.Q = np.zeros((T+2, X_MAX+1, U_MAX+1), dtype=float)

    def choose_action(self, t, x):
        # Epsilon-greedy action selection
        if t > self.T:
            return 0
        if rng.random() < self.epsilon:
            return int(rng.integers(0, self.U_MAX+1))
        return int(np.argmin(self.Q[t, x, :]))

    def learn(self, t, x, u, cost, next_state, done):
        alpha = self.lr
        t_next, x_next = next_state
        # YOUR CODE HERE
        # Update the Q-value using the Q-learning update rule
        q_old = self.Q[t, x, u]
        if done:
            target = cost
        else:
            target = cost + self.gamma * np.min(self.Q[t_next, x_next, :])
        self.Q[t, x, u] = (1 - alpha) * q_old + alpha * target
        # END YOUR CODE
        # Decay epsilon and learning rate
        self.epsilon = max(self.eps_min, self.epsilon - self.eps_dec)
        self.lr = max(self.lr_min, self.lr - self.lr_dec)

    def greedy_policy(self):
        return np.argmin(self.Q[1:self.T+1, :, :], axis=2).T

    def cost_to_go(self):
        V = np.min(self.Q[1:self.T+1, :, :], axis=2).T
        J_hat = V
        return J_hat


**Parameters Known to the Agent:**
- Horizon: $T=12$ hours (8am–8pm).
- State $x_t\in [0, 70]$: patients waiting at start of hour $t$.
- Action $u_t \in [0, 20]$: number of **on‑demand** doctors to add in hour $t$.
- Base doctors: $10$ permanently scheduled (not part of control; cost sunk).
- Service: each doctor (base or on‑demand) treats **2 patients/hour**.

In [9]:
T, X_MAX, U_MAX = 12, 70, 20

env = HospitalEnvQL(T=T, X_MAX=X_MAX, U_MAX=U_MAX, lam=LAM, w=w, W=W, wage=WAGE)
agent = QLearningAgent(T=T, X_MAX=X_MAX, U_MAX=U_MAX)

def train(agent, env, n_episodes=20000, log_every=4000):
    scores = []
    for ep in range(1, n_episodes+1):
        state = env.reset()
        done = False
        total_cost = 0.0
        while not done:
            t, x = state
            u = agent.choose_action(t, x)
            next_state, cost, done = env.step(u)
            agent.learn(t, x, u, cost, next_state, done)
            state = next_state
            total_cost += cost
        scores.append(total_cost)
        if ep % log_every == 0:
            avg = np.mean(scores[-1000:])
            print(f"Episode {ep:6d} | avg cost (last 1000): {avg:,.1f} | eps={agent.epsilon:.3f} lr={agent.lr:.3f}")
    return scores

scores = train(agent, env, n_episodes=4000, log_every=400)


Episode    400 | avg cost (last 1000): 36,735.8 | eps=0.040 lr=0.298
Episode    800 | avg cost (last 1000): 27,482.4 | eps=0.020 lr=0.295
Episode   1200 | avg cost (last 1000): 19,977.3 | eps=0.020 lr=0.293
Episode   1600 | avg cost (last 1000): 18,360.8 | eps=0.020 lr=0.290


Episode   2000 | avg cost (last 1000): 16,684.3 | eps=0.020 lr=0.288
Episode   2400 | avg cost (last 1000): 15,024.8 | eps=0.020 lr=0.286
Episode   2800 | avg cost (last 1000): 13,772.9 | eps=0.020 lr=0.283


Episode   3200 | avg cost (last 1000): 13,604.0 | eps=0.020 lr=0.281
Episode   3600 | avg cost (last 1000): 13,172.6 | eps=0.020 lr=0.278
Episode   4000 | avg cost (last 1000): 12,189.8 | eps=0.020 lr=0.276


### Learning Curve Visualization

In [10]:
plt.figure(figsize=(7,4))
win = 200
if len(scores) >= win:
    ma = np.convolve(scores, np.ones(win)/win, mode='valid')
    plt.plot(np.arange(len(ma))+win-1, ma, label=f"Moving avg ({win})")
plt.plot(scores, alpha=0.3, label="Episode cost")
plt.xlabel("Episode"); plt.ylabel("Cost"); plt.title("Q-Learning: Learning Curve"); plt.legend(); plt.show()


/var/folders/70/8g8pbwhx56z6cbq_qh7gxzbm0000gn/T/ipykernel_90283/3513748541.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("Episode"); plt.ylabel("Cost"); plt.title("Q-Learning: Learning Curve"); plt.legend(); plt.show()


The plot above shows how the Q-learning agent’s performance evolves over training:

* **Orange line (Episode cost):**  
  The total cost collected in each training episode. This is very noisy because arrivals are random and the policy is still learning.

* **Blue line (Moving average over 200 episodes):**  
  A smoothed version of the costs, which highlights the long-term trend in learning.  
  - At the start, costs are very low (high costs) because the agent explores randomly.  
  - Over time, the moving average stabilizes, indicating that the agent has converged to a better policy that reduces expected costs.

This visualization helps confirm that the Q-learning updates are improving the policy, even though individual episode outcomes fluctuate significantly due to randomness in arrivals.


### Visualize the Sliced Q-Value Table

- **Q-value heatmaps:** show $Q_t(x_t,u_t)$ for selected stages.  
  - x-axis: action $u_t$ (surge doctors)  
  - y-axis: queue length $x_t$  
  - color: estimated Q-value (cumulative expected cost)  
<!-- 
- **Greedy policy plots:** show $\pi_t(x_t)=\arg\min_{u_t}Q_t(x_t,u_t)$ for the same stages.  
  - x-axis: queue length $x_t$  
  - y-axis: optimal action $u_t$ suggested by the learned Q-table   -->


In [11]:
time_slices = [1, 4, 8, 11]

# --- Q-value heatmaps ---
fig, axes = plt.subplots(1, len(time_slices), figsize=(18, 5))
for i, t in enumerate(time_slices):
    q_slice = agent.Q[t, :, :]   # shape (X_MAX+1, U_MAX+1)
    im = axes[i].imshow(q_slice, aspect="auto", origin="lower")
    axes[i].set_title(f"Q-Table(t={t})")
    axes[i].set_xlabel("Action u (surge doctors)")
    axes[i].set_ylabel("Queue length x")
    fig.colorbar(im, ax=axes[i])
plt.tight_layout()
plt.show()

/var/folders/70/8g8pbwhx56z6cbq_qh7gxzbm0000gn/T/ipykernel_90283/2488231062.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Q-Table and Greedy Policy Slices**

The top row shows **slices of the final converged Q-value table** $Q_t(x_t,u_t)$ at $t=1,4,8,11$.  
The bottom row shows the corresponding **greedy policies** $\pi_t(x_t)=\arg\min_{u_t}Q_t(x_t,u_t)$ derived from these Q-values.

**Structure of Q-values.**  
   - In each slice, Q-values grow with queue length $x_t$ and with larger actions $u_t$, since both waiting patients and surge doctors contribute to cost.  
   - For small queues, low $u_t$ is optimal; for large queues, higher $u_t$ becomes cost-effective. 

**Implications.**  
These figures confirm that after convergence, Q-learning has learned a *stage-dependent policy*:  
- Early in the horizon, the agent is conservative about hiring extra doctors.  
- As the horizon shortens, the policy becomes more aggressive at higher queue lengths.


### Extracting the Learned Policy and Cost-to-Go

After training, we extract two key results from the Q-learning agent:

1. **Policy table (`policy_df`)**  
   - `agent.greedy_policy()` takes the learned Q-values and computes the greedy policy:  
     $$
     \pi_t(x_t) = \arg\min_{u_t} Q_t(x_t,u_t).
     $$  
   - Each entry in the table is the number of on-demand doctors $u_t$ that the agent would choose, given queue length $x$ and hour $t$.

2. **Estimated cost-to-go (`J_df`)**  
   - `agent.cost_to_go()` computes the value of each state as  
     $$
     \hat J_t(x_t) = \min_{u_t} Q_t(x_t,u_t),
     $$  
   - Each entry in the table is the agent’s estimate of the minimum future cost starting from $(t, x)$.

Both results are stored as Pandas DataFrames with:
- **Rows:** queue lengths $x_t = 0,1,\dots,X_{\max}$.  
- **Columns:** hours $t = 1,\dots,T$.  

These tables provide a tabular summary of the learned decision rules and their associated cost estimates, which we can later visualize as heatmaps.


In [12]:
policy = agent.greedy_policy()
J_hat = agent.cost_to_go()

hours = np.arange(1, T+1)
states = np.arange(0, X_MAX+1)

QL_policy_df = pd.DataFrame(policy, index=states, columns=hours)
QL_J_df = pd.DataFrame(J_hat, index=states, columns=hours)

display(QL_policy_df.head(3))
display(QL_J_df.head(3))


,1,2,3,4,5,6,7,8,9,10,11,12
0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,1


,1,2,3,4,5,6,7,8,9,10,11,12
0,693.656624,559.754532,572.929942,649.915597,535.395285,487.460813,524.008692,490.159359,459.086901,458.551851,479.174186,406.915080
1,784.495835,654.555315,600.026788,562.422718,494.627179,416.225413,495.927314,453.183256,410.564794,388.581893,698.035955,319.352722
2,795.898036,735.637987,684.839414,695.264993,551.302721,452.108487,554.241361,494.062295,493.339126,475.291720,208.858041,1400.428737


### Visualizing the Policy and Cost-to-Go using Heatmaps for Q-Learning

While the tables provide the exact numeric values, heatmaps make it easier to **see the structure** of the learned policy and costs across all states and hours:

1. **Policy Heatmap**  
   - Colors represent the number of on-demand doctors $u_t^*(x_t)$ chosen by the greedy policy.  
   - Horizontal axis: hours of the day ($t=1,\dots,12$).  
   - Vertical axis: queue length at the start of the hour ($x_t=0,\dots,70$).  
   - Lighter colors typically indicate hiring more surge doctors.  
   - Interpretation: shows how staffing decisions escalate with larger queues or as the day progresses.

2. **Cost-to-Go Heatmap**  
   - Colors represent the estimated cost-to-go $\hat J_t(x_t)$.  
   - Horizontal axis: hours.  
   - Vertical axis: queue length.  
   - Larger values correspond to higher expected future costs.  
   - Interpretation: illustrates how the “price” of being in a given state changes over time and with backlog.

Together, these heatmaps provide an intuitive visualization of the **policy structure** and the **expected costs** that drive it.


In [13]:

plt.figure(figsize=(8,5))
plt.imshow(QL_policy_df.values, aspect='auto', origin='lower')
plt.colorbar()
plt.yticks(ticks=np.linspace(0, X_MAX, 9), labels=[str(int(v)) for v in np.linspace(0, X_MAX, 9)])
plt.xticks(ticks=np.arange(T), labels=[str(t) for t in range(1, T+1)])
plt.xlabel("Hour t (1=8–9am,…,12=7–8pm)"); plt.ylabel("Queue length x"); plt.title("Policy Heatmap: u_t*(x)")
plt.show()

plt.figure(figsize=(8,5))
plt.imshow(QL_J_df.values, aspect='auto', origin='lower')
plt.colorbar()
plt.yticks(ticks=np.linspace(0, X_MAX, 9), labels=[str(int(v)) for v in np.linspace(0, X_MAX, 9)])
plt.xticks(ticks=np.arange(T), labels=[str(t) for t in range(1, T+1)])
plt.xlabel("Hour t"); plt.ylabel("Queue length x"); plt.title("Estimated Cost-to-Go Ĵ_t(x)")
plt.show()


/var/folders/70/8g8pbwhx56z6cbq_qh7gxzbm0000gn/T/ipykernel_90283/2495282174.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/70/8g8pbwhx56z6cbq_qh7gxzbm0000gn/T/ipykernel_90283/2495282174.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Compare **Backward Induction** and **Q-Learning** Results

After running both methods on the hospital surge problem, we can compare their outputs.

**Policy Tables.**  
- **Backward Induction (BI):** produces the exact optimal policy $\pi_t^{\text{BI}}(x_t)$ for every stage $t$ and queue length $x_t$. The BI policy is guaranteed to minimize expected total cost given the model.  
- **Q-Learning (QL):** produces an approximate policy $\pi_t^{\text{QL}}(x_t)$ by taking $\arg\min_{u_t} Q_t(x_t,u_t)$. Early in training, this policy may be noisy or suboptimal, but as the number of episodes grows, $\pi^{\text{QL}}$ converges toward $\pi^{\text{BI}}$ in the frequently visited states.

**Cost-to-Go Tables.**  
- **BI cost-to-go $J_t(x_t)$:** exact expected cumulative cost under the optimal policy. This serves as the benchmark.  
- **QL cost-to-go $\hat{J}_t(x_t)$:** estimated by $\hat{J}_t(x_t) = \min_{u_t} Q_t(x_t,u_t)$. Initially it may deviate from $J_t(x_t)$, especially for rarely visited states, but with sufficient exploration and learning the estimates become close to the BI benchmark.

**Summary of Differences.**
- BI yields exact, model-based results but requires full knowledge of the transition and cost functions and heavy computation across all states.  
- QL learns from interaction, is more scalable to unknown or complex environments, and gradually approximates the BI benchmark in terms of both policy and cost-to-go.

In [14]:
# --- Policy comparison ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(BI_policy_df.values.T, aspect="auto", origin="lower")
axes[0].set_title("BI Policy (u*_t(x))")
axes[0].set_xlabel("Hour t")
axes[0].set_ylabel("Queue length x")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(QL_policy_df.values.T, aspect="auto", origin="lower")
axes[1].set_title("QL Policy (π_t(x))")
axes[1].set_xlabel("Hour t")
axes[1].set_ylabel("Queue length x")
fig.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

# --- Cost-to-go comparison ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im2 = axes[0].imshow(BI_cost_df.values.T, aspect="auto", origin="lower")
axes[0].set_title("BI Cost-to-Go J_t(x)")
axes[0].set_xlabel("Hour t")
axes[0].set_ylabel("Queue length x")
fig.colorbar(im2, ax=axes[0])

im3 = axes[1].imshow(QL_J_df.values.T, aspect="auto", origin="lower")
axes[1].set_title("QL Estimated Cost-to-Go Ĵ_t(x)")
axes[1].set_xlabel("Hour t")
axes[1].set_ylabel("Queue length x")
fig.colorbar(im3, ax=axes[1])

plt.tight_layout()
plt.show()


/var/folders/70/8g8pbwhx56z6cbq_qh7gxzbm0000gn/T/ipykernel_90283/2871363947.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/70/8g8pbwhx56z6cbq_qh7gxzbm0000gn/T/ipykernel_90283/2871363947.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


***Why Do the BI and QL Cost-to-Go Heatmaps Look Different?***

Looking at the heatmaps, we see that the **Backward Induction (BI)** results are smooth and complete, while the **Q-Learning (QL)** results appear patchy or irregular in some regions.

**Intuition:**
- **BI heatmaps** are derived from a full dynamic program with a known transition model. Every state $(t,x_t)$ is computed exactly, even if the state is rarely visited. This produces a smooth cost-to-go surface and a complete policy table.  
- **QL heatmaps** are learned from sampled experience. The agent only updates $Q_t(x_t,u_t)$ when it visits state $x_t$ at time $t$ and tries action $u_t$.  
  - For **frequently visited states** (e.g., small queue lengths, early hours), QL gathers many updates, so the estimated cost-to-go aligns closely with BI.  
  - For **rarely visited or extreme states** (e.g., very large queues, late hours), QL has fewer samples. These states may remain under-explored, leading to noisy or unstable estimates, which show up as irregular patches in the heatmap.  

**Key takeaway:**  
BI produces a complete benchmark solution by exhaustive computation, while QL focuses learning on the trajectories it actually experiences. This makes QL more practical in large or unknown systems, but its estimated policy and cost-to-go may differ in under-explored areas.


---
## Part III: SARSA, and What "On-Policy" Actually Means

Q-learning bootstraps from `np.min(self.Q[t_next, x_next, :])`: it assumes that next
period we will act **greedily**, taking the cheapest action available.

But `choose_action` is epsilon-greedy. Some of the time we will *not* act greedily,
we will do something random. So Q-learning is valuing a policy that is not the one it
is actually following. That is what **off-policy** means.

SARSA fixes this by bootstrapping from the action it will **actually take next**.
That is the entire difference, and it is one line.

In [15]:
class SARSAAgent(QLearningAgent):
    """
    Identical to QLearningAgent in every respect except the bootstrap term.

    We inherit choose_action, greedy_policy and cost_to_go unchanged, so the ONLY
    difference between the two algorithms is the single marked line in learn().

    Note the extra argument: SARSA needs to know which action it will take next,
    so learn() takes u_next. Q-learning does not need it, because min() over the
    next row does not depend on what we actually do.
    """

    def learn(self, t, x, u, cost, next_state, u_next, done):
        alpha = self.lr
        t_next, x_next = next_state
        q_old = self.Q[t, x, u]
        if done:
            target = cost
        else:
            # ---- THE ONE LINE THAT DIFFERS FROM Q-LEARNING -------------------
            # Q-learning: target = cost + self.gamma*np.min(self.Q[t_next, x_next, :])
            # SARSA:      bootstrap from the action actually taken next
            target = cost + self.gamma * self.Q[t_next, x_next, u_next]
            # ------------------------------------------------------------------
        self.Q[t, x, u] = (1 - alpha) * q_old + alpha * target
        # Decay epsilon and learning rate (identical to QLearningAgent)
        self.epsilon = max(self.eps_min, self.epsilon - self.eps_dec)
        self.lr = max(self.lr_min, self.lr - self.lr_dec)

The training loop also has to change shape. Q-learning can update as soon as it sees
the next state. SARSA has to choose the next action **first**, then update, then use
that same action. This is where the name comes from: State, Action, Reward, State, Action.

In [16]:
def train_sarsa(agent, env, n_episodes=4000, log_every=400):
    scores = []
    for ep in range(1, n_episodes+1):
        t, x = env.reset()
        u = agent.choose_action(t, x)          # choose the FIRST action up front
        done = False
        total_cost = 0.0
        while not done:
            next_state, cost, done = env.step(u)
            t_next, x_next = next_state
            # choose the next action BEFORE learning, because SARSA bootstraps from it
            u_next = agent.choose_action(t_next, x_next) if not done else 0
            agent.learn(t, x, u, cost, next_state, u_next, done)
            total_cost += cost
            t, x, u = t_next, x_next, u_next   # and then actually take it
        scores.append(total_cost)
        if ep % log_every == 0:
            print(f"Episode {ep:6d} | avg cost (last 1000): {np.mean(scores[-1000:]):,.1f} "
                  f"| eps={agent.epsilon:.3f} lr={agent.lr:.3f}")
    return scores


env_s   = HospitalEnvQL(T=T, X_MAX=X_MAX, U_MAX=U_MAX, lam=LAM, w=w, W=W, wage=WAGE)
agent_s = SARSAAgent(T=T, X_MAX=X_MAX, U_MAX=U_MAX)
scores_s = train_sarsa(agent_s, env_s, n_episodes=4000, log_every=400)

Episode    400 | avg cost (last 1000): 39,760.3 | eps=0.040 lr=0.298
Episode    800 | avg cost (last 1000): 29,389.8 | eps=0.020 lr=0.295
Episode   1200 | avg cost (last 1000): 21,921.3 | eps=0.020 lr=0.293
Episode   1600 | avg cost (last 1000): 20,994.6 | eps=0.020 lr=0.290


Episode   2000 | avg cost (last 1000): 19,855.0 | eps=0.020 lr=0.288
Episode   2400 | avg cost (last 1000): 18,346.7 | eps=0.020 lr=0.286
Episode   2800 | avg cost (last 1000): 17,883.1 | eps=0.020 lr=0.283
Episode   3200 | avg cost (last 1000): 17,218.7 | eps=0.020 lr=0.281


Episode   3600 | avg cost (last 1000): 16,184.8 | eps=0.020 lr=0.278
Episode   4000 | avg cost (last 1000): 15,391.6 | eps=0.020 lr=0.276


### Comparing the two learning curves

In [17]:
def moving_avg(v, k=200):
    v = np.asarray(v, dtype=float)
    return np.convolve(v, np.ones(k)/k, mode="valid")

plt.figure(figsize=(7,4))
plt.plot(moving_avg(scores),   label="Q-learning (off-policy)")
plt.plot(moving_avg(scores_s), label="SARSA (on-policy)")
plt.xlabel("Episode"); plt.ylabel("Total cost per episode (moving average)")
plt.title("Q-learning vs SARSA on the hospital problem")
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

print(f"Q-learning, mean cost over last 500 episodes: {np.mean(scores[-500:]):,.0f}")
print(f"SARSA,      mean cost over last 500 episodes: {np.mean(scores_s[-500:]):,.0f}")

Q-learning, mean cost over last 500 episodes: 11,588
SARSA,      mean cost over last 500 episodes: 15,473


/var/folders/70/8g8pbwhx56z6cbq_qh7gxzbm0000gn/T/ipykernel_90283/2881923430.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### An honest note on what you are looking at

**Do not expect these two curves to separate much on this problem.** They will not, and
that is worth understanding rather than glossing over.

On-policy and off-policy learners diverge when **exploring is dangerous**: when a random
action can drop you somewhere catastrophic and unrecoverable. The classic example is a
cliff, where the optimal route runs right along the edge, and an on-policy learner
deliberately walks a wider, safer path because it prices in its own random steps.

A hospital staffing hour is not a cliff. If you staff badly for one hour you pay for it
and the queue recovers. Nothing is unrecoverable. So both algorithms end up in
essentially the same place.

The place to see the difference is the 1-D driving example from lecture, where position 0
is a genuine sinkhole sitting right beside the optimal path. Even there the *policies*
come out the same, because there is no alternative route. What differs is the
**Q-values**: with a fixed epsilon, SARSA's stay systematically pessimistic because they
include the cost of its own future random moves, while Q-learning's converge to the exact
optimum regardless of epsilon.

That is the answer to A2 (j) and (k): Q-learning with fixed epsilon converges to the
optimal Q, SARSA with epsilon decaying to zero converges to the optimal Q, and SARSA with
a *fixed* epsilon does not, because it keeps valuing the exploratory policy it is
actually following.

### The driving example from lecture, run three ways

Small enough to converge in seconds, so you can see the claim above rather than take it
on faith. Reward convention here (so `max`, not `min`), matching the lecture slides.

In [18]:
def track_run(algo, eps0, decay_to_zero, n_ep=200000, alpha=0.1, gamma=0.9, seed=1):
    """1-D track: 0 = sinkhole (-10), 4 = goal (+10), each move -1. Spawn at 2."""
    r_ = np.random.default_rng(seed)
    Q = np.zeros((5, 2))                      # rows 1..3 are the real states
    eps = eps0

    def pick(s, e):
        return r_.integers(0, 2) if r_.random() < e else int(np.argmax(Q[s]))

    for ep in range(n_ep):
        if decay_to_zero:
            eps = eps0 * max(0.0, 1 - ep / (0.8 * n_ep))
        s = 2
        a = pick(s, eps)
        while True:
            s2 = s - 1 if a == 0 else s + 1
            r = 10.0 if s2 == 4 else (-10.0 if s2 == 0 else -1.0)
            terminal = s2 in (0, 4)
            if terminal:
                target = r
            else:
                a2 = pick(s2, eps)
                target = r + gamma * (np.max(Q[s2]) if algo == "Q" else Q[s2, a2])
            Q[s, a] += alpha * (target - Q[s, a])
            if terminal:
                break
            s, a = s2, a2
    return Q


exact = {(1,0): -10.0, (1,1): 6.2, (2,0): 4.58, (2,1): 8.0, (3,0): 6.2, (3,1): 10.0}
runs = [("Q-learning, fixed eps=0.1", track_run("Q", 0.1, False)),
        ("SARSA,      fixed eps=0.1", track_run("S", 0.1, False)),
        ("SARSA,      eps -> 0",      track_run("S", 0.1, True))]

hdr = "  ".join(f"Q({s},{'LR'[a]})" for (s, a) in exact)
print(f"{'':<26} {hdr}")
print(f"{'exact optimal Q*':<26} " + "  ".join(f"{exact[k]:>7.2f}" for k in exact))
print("-" * 78)
for name, Qm in runs:
    print(f"{name:<26} " + "  ".join(f"{Qm[s,a]:>7.2f}" for (s, a) in exact))

                           Q(1,L)  Q(1,R)  Q(2,L)  Q(2,R)  Q(3,L)  Q(3,R)
exact optimal Q*            -10.00     6.20     4.58     8.00     6.20    10.00
------------------------------------------------------------------------------
Q-learning, fixed eps=0.1   -10.00     6.20     4.58     8.00     6.20    10.00
SARSA,      fixed eps=0.1   -10.00     5.75     3.67     8.00     5.55    10.00
SARSA,      eps -> 0        -10.00     6.18     4.56     8.00     5.72    10.00


Read the **`Q(1,R)`** and **`Q(2,L)`** columns, whose exact optimal values are 6.20 and 4.58.

- **Q-learning with fixed epsilon** hits both exactly. Off-policy: the `max` in its update
  targets the greedy action, so how randomly it *behaves* does not change what it *learns*.
- **SARSA with fixed epsilon** falls short on both (about 5.75 and 3.67). On-policy: it is
  correctly valuing a policy that takes a random action 10% of the time, and that policy
  really is worse.
- **SARSA with epsilon decaying to zero** recovers them (6.18, 4.56), because the policy it
  follows converges to the greedy one.

Notice that **not every cell separates**. `Q(2,R)` and `Q(3,R)` come out at 8.00 and 10.00
for all three, because they bootstrap from a state where the greedy action is taken almost
every time, so on-policy and off-policy targets nearly coincide there. The gap opens up
only where the next state has a genuinely bad alternative close by.

`Q(3,L)` is the one cell where SARSA-with-decay still reads low (5.72 against 6.20). That is
a visitation artifact, not a convergence failure: once epsilon reaches zero the agent almost
never takes Left at position 3, so that cell stops being updated before it finishes settling.

Nobody's code is broken. The three algorithms are answering three slightly different
questions.